# RQ1: Attribution Stability Benchmark Under Meaning-Preserving Perturbations
Full-scale evaluation of token attribution robustness across 16 lexical, syntactic, and discourse perturbations on SST-2 (`Qwen/Qwen2.5-7B` & `Qwen/Qwen2.5-3B`) using empirically calibrated Pareto-optimal hyperparameters:
- **Occlusion (Leave-One-Out)**: deterministic ($L$ forward passes)
- **Gradient $\times$ Input**: deterministic ($1$ forward + $1$ backward pass)
- **Integrated Gradients**: $m = 100$ Riemann steps
- **LIME**: $N = 1000$ neighborhood coalitions
- **SHAP (Permutation)**: $B = 2500$ evaluation budget

### 1. GPU Environment Verification

In [ ]:
!nvidia-smi
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB) | bf16: {torch.cuda.is_bf16_supported()}")

### 2. Google Drive Mounting, Repository Setup, and Dependencies

In [ ]:
import os

IN_COLAB = "google.colab" in str(get_ipython())
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_RESULTS = "/content/drive/MyDrive/llm_xai_results"
    os.makedirs(DRIVE_RESULTS, exist_ok=True)

    GITHUB_TOKEN = "" #@param {type:"string"}
    REPO_OWNER_REPO = "jakubstachyra/llm-xai" #@param {type:"string"}
    BRANCH = "main" #@param {type:"string"}

    try:
        from google.colab import userdata
        token_from_secrets = userdata.get('GH_TOKEN')
        if token_from_secrets:
            GITHUB_TOKEN = token_from_secrets
    except Exception:
        pass

    if GITHUB_TOKEN.strip():
        CLONE_URL = f"https://{GITHUB_TOKEN.strip()}@github.com/{REPO_OWNER_REPO}.git"
    else:
        CLONE_URL = f"https://github.com/{REPO_OWNER_REPO}.git"

    REPO_DIR = "/content/llm-xai"
    if not os.path.exists(REPO_DIR):
        res = !git clone -b {BRANCH} {CLONE_URL} {REPO_DIR}
        print("\n".join(res))
        if not os.path.exists(REPO_DIR):
            raise RuntimeError("Cloning error! Provide Personal Access Token (PAT) in GITHUB_TOKEN or set GH_TOKEN secret.")
        %cd {REPO_DIR}
    else:
        %cd {REPO_DIR}
        !git checkout {BRANCH}
        !git pull origin {BRANCH}

    !pip install -q --upgrade numpy "transformers>=4.43.0,<4.47.0" "accelerate>=0.30.0" "datasets>=2.19.0" "shap>=0.46.0" scikit-learn scipy pandas matplotlib seaborn huggingface_hub

    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN')
        if hf_token:
            os.environ["HF_TOKEN"] = hf_token
    except Exception:
        pass
else:
    DRIVE_RESULTS = "results"
    os.makedirs(DRIVE_RESULTS, exist_ok=True)
    print(f"Running locally. Results directory: {os.path.abspath(DRIVE_RESULTS)}")

### 3. Pipeline Smoke Test (2 samples)
Quick verification that the selected Qwen model, tokenizer, perturbations, and all 5 explainers execute cleanly.

In [ ]:
smoke_model = "Qwen/Qwen2.5-7B" #@param ["Qwen/Qwen2.5-7B", "Qwen/Qwen2.5-3B", "microsoft/phi-2"]

!python scripts/run_rq1.py \
    --model {smoke_model} \
    --samples 2 \
    --methods occlusion,grad_x_input,ig,lime,shap \
    --max-evals 100 \
    --lime-samples 100 \
    --ig-steps 10 \
    --output-dir /tmp/smoke_test_rq1 \
    --no-resume

### 4. Main RQ1 Stability Experiment (Milestone Auto-Resume: 300 -> 500 -> 872)
- Automatically saves checkpoints to Google Drive every 10 samples (`rq1_stability_raw.csv`).
- Prefix-preserving sampling (`seed=42`) ensures that once `N=300` finishes, it seamlessly continues from sample `300` to `500`, and then from `500` to the full validation set (`872`).
- If interrupted at any time, re-running this cell resumes from the exact last saved checkpoint.

In [ ]:
#@title Main RQ1 Stability Parameters
model = "Qwen/Qwen2.5-7B" #@param ["Qwen/Qwen2.5-7B", "Qwen/Qwen2.5-3B", "microsoft/phi-2"]
milestones = "300,500" #@param ["300,500", "300", "500", "872"]
seed = 42 #@param {type:"integer"}
methods = "occlusion,grad_x_input,ig,lime,shap" #@param {type:"string"}

shap_max_evals = 1200 #@param {type:"integer"}
lime_samples = 500 #@param {type:"integer"}
ig_steps = 50 #@param {type:"integer"}

max_run_samples = 0 #@param {type:"integer"}
no_noise_floor = False #@param {type:"boolean"}
auto_shutdown = True #@param {type:"boolean"}

tag = model.split("/")[-1]
output_dir = f"{DRIVE_RESULTS}/rq1_stability_{tag}"
limit_flag = f"--max-run-samples {max_run_samples}" if max_run_samples > 0 else ""
noise_flag = "--no-noise-floor" if no_noise_floor else ""

target_list = [int(x.strip()) for x in milestones.split(",") if x.strip()]

for target_n in target_list:
    print(f"\n{'='*75}")
    print(f"  RQ1 STABILITY MILESTONE: N = {target_n} | MODEL: {model}")
    print(f"  Output Directory: {output_dir}")
    print(f"{'='*75}\n")
    !python scripts/run_rq1.py \
        --model {model} \
        --samples {target_n} \
        --seed {seed} \
        --methods {methods} \
        --max-evals {shap_max_evals} \
        --lime-samples {lime_samples} \
        --ig-steps {ig_steps} \
        {limit_flag} \
        {noise_flag} \
        --output-dir {output_dir}

if auto_shutdown and IN_COLAB:
    print("Releasing GPU and stopping Colab billing...")
    from google.colab import runtime
    runtime.unassign()

### 5. Results Summary Table (Overall Stability & Noise Floor)

In [ ]:
import os
import pandas as pd
from IPython.display import display

raw_csv = os.path.join(output_dir, "rq1_stability_raw.csv")
if os.path.exists(raw_csv):
    df_raw = pd.read_csv(raw_csv)
    main_df = df_raw[(df_raw["perturbation_category"] != "control") & (df_raw["is_preserved"] == True)]
    ctrl_df = df_raw[df_raw["perturbation_category"] == "control"]

    overall = main_df.groupby("explainer_name").agg(
        n_pairs=("spearman", "size"),
        n_samples=("sample_id", "nunique"),
        spearman_mean=("spearman", "mean"),
        spearman_sem=("spearman", "sem"),
        kendall_mean=("kendall", "mean"),
        jaccard_k1_mean=("jaccard_k1", "mean"),
        jaccard_k3_mean=("jaccard_k3", "mean"),
        jaccard_k5_mean=("jaccard_k5", "mean"),
        time_s_mean=("orig_time_s", "mean"),
        fwd_passes_mean=("orig_forward_passes", "mean"),
        bwd_passes_mean=("orig_backward_passes", "mean"),
        vram_mb_max=("vram_mb", "max")
    ).reset_index()

    if len(ctrl_df) > 0:
        floor = ctrl_df.groupby("explainer_name").agg(
            noise_floor_spearman=("spearman", "mean"),
            noise_floor_jaccard_k3=("jaccard_k3", "mean")
        ).reset_index()
        overall = overall.merge(floor, on="explainer_name", how="left")

    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", 1000)
    display(overall.round(3))
else:
    print(f"Raw file {raw_csv} does not exist yet.")

### 6. Stability Across Perturbation Categories & Boxplot Visualization

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if os.path.exists(raw_csv):
    df_raw = pd.read_csv(raw_csv)
    main_df = df_raw[(df_raw["perturbation_category"] != "control") & (df_raw["is_preserved"] == True)]

    # Category breakdown table
    cat_piv = main_df.pivot_table(
        index="explainer_name",
        columns="perturbation_category",
        values="spearman",
        aggfunc="mean"
    ).round(3)
    print("Mean Spearman Stability by Perturbation Category:")
    display(cat_piv)

    plt.figure(figsize=(10, 5.5), dpi=150)
    sns.boxplot(data=main_df, x="explainer_name", y="spearman", hue="perturbation_category")
    plt.title(f"RQ1: Attribution Ranking Stability by Perturbation Category – {tag}", fontweight="bold")
    plt.xlabel("XAI Method")
    plt.ylabel(r"Spearman Rank Correlation $\rho$")
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.legend(title="Category", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "rq1_category_boxplot.png"))
    plt.show()

### 7. Manual Session Teardown (Stop Compute Units Billing)

In [ ]:
if IN_COLAB:
    from google.colab import runtime
    print("Disconnecting Colab runtime...")
    runtime.unassign()